# MeriAwaaz AI — Supervisor Routing Tests
Tests the supervisor routing decision for text and dashboard_refresh inputs.

In [ ]:
from app.core.llm import get_model
from app.supervisor import build_workflow
print('imports OK')

## Stub agents
Each stub prints which node was reached and passes messages through unchanged.

In [ ]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_core.messages import AIMessage

def make_stub(name: str):
    def node_fn(state: MessagesState):
        print(f'[STUB REACHED] {name}')
        return {"messages": [AIMessage(content=f"stub: {name} completed")]}
    g = StateGraph(MessagesState)
    g.add_node(name, node_fn)
    g.add_edge(START, name)
    g.add_edge(name, END)
    return g.compile()

citizen_intelligence_agent   = make_stub('citizen_intelligence_agent')
demand_intelligence_agent    = make_stub('demand_intelligence_agent')
knowledge_fusion_agent       = make_stub('knowledge_fusion_agent')
policy_recommendation_agent  = make_stub('policy_recommendation_agent')
explainability_agent         = make_stub('explainability_agent')
print('stubs ready')

## Build supervised workflow

In [ ]:
orchestrator = build_workflow([
    citizen_intelligence_agent,
    demand_intelligence_agent,
    knowledge_fusion_agent,
    policy_recommendation_agent,
    explainability_agent,
])
print('workflow compiled')

## Test 1 — English text input
Expected: routes to citizen_intelligence_agent

In [ ]:
from langchain_core.messages import HumanMessage

result = orchestrator.invoke(
    {"messages": [HumanMessage(content="Our village school has no toilets and children are dropping out.")]},
    config={"configurable": {"thread_id": "test-text-1"}},
)
print('--- messages ---')
for m in result['messages']:
    print(f'{m.__class__.__name__}: {m.content[:120]}')

## Test 2 — Dashboard refresh
Expected: routes to demand_intelligence_agent

In [ ]:
result2 = orchestrator.invoke(
    {"messages": [HumanMessage(content="dashboard refresh")]},
    config={"configurable": {"thread_id": "test-dash-1"}},
)
print('--- messages ---')
for m in result2['messages']:
    print(f'{m.__class__.__name__}: {m.content[:120]}')

## Test 3 — Hindi text input
Expected: routes to citizen_intelligence_agent

In [ ]:
result3 = orchestrator.invoke(
    {"messages": [HumanMessage(content="हमारे गाँव में सड़क बहुत خराब है।")]},
    config={"configurable": {"thread_id": "test-hindi-1"}},
)
print('--- messages ---')
for m in result3['messages']:
    print(f'{m.__class__.__name__}: {m.content[:120]}')